In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.integrate import simpson
from scipy.sparse.linalg import eigsh
from scipy.sparse import diags
from scipy.interpolate import CubicSpline
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

# =============================================
# Parámetros Físicos (Optimizados)
# =============================================
D0 = 2e-9           # Coeficiente de difusión libre [m²/s]
R = 1e-6            # Radio del cilindro [m]
L = 100e-9          # Longitud axial del dominio [m]
z_c = 10e-9         # Altura crítica de transición [m]
lambda_param = 0.2  # Parámetro de decaimiento [nm⁻¹]
kappa = 0.0         # Tasa de adsorción superficial [s⁻¹]
rho0 = 0.5e-6       # Posición radial inicial [m]
z0 = 5e-9           # Posición axial inicial [m]
n_modes_radial = 16 # Modos radiales
m_max = 25          # Modos axiales considerados
N_z = 1000          # Nodos en z (malla refinada)
alpha_grid = 6.0    # Parámetro de densidad de malla
delta = 0.5e-9      # Ancho de transición para D(z)

# =============================================
# 1. Generación de Malla Adaptativa en z
# =============================================
j = np.arange(N_z)
z = z_c * (1 + np.sinh(alpha_grid * j / (N_z - 1)) / np.sinh(alpha_grid))

# =============================================
# 2. Perfil de Coeficiente de Difusión D(z)
# =============================================
def diffusion_coefficient(z):
    S = 0.5 * (1 + np.tanh((z - z_c)/delta))
    return D0 * (np.exp(-lambda_param * z) * (1 - S) + S)

D_values = diffusion_coefficient(z)

# =============================================
# 3. Solución Radial (Funciones de Bessel)
# =============================================
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros / R
norm_J0_sq = (R**2 / 2) * jv(1, alpha_zeros)**2

# =============================================
# 4. Solución Axial Corregida
# =============================================
eigenvalues_dict = {}
eigenvectors_dict = {}

for n in range(n_modes_radial):
    main_diag = np.zeros(N_z)
    upper_diag = np.zeros(N_z - 1)
    lower_diag = np.zeros(N_z - 1)
    
    for j in range(1, N_z - 1):
        h_j_prev = z[j] - z[j-1]
        h_j_next = z[j+1] - z[j]
        
        D_j_plus_half = 2 * D_values[j] * D_values[j+1] / (D_values[j] + D_values[j+1] + 1e-30)
        D_j_minus_half = 2 * D_values[j-1] * D_values[j] / (D_values[j-1] + D_values[j] + 1e-30)
        
        coeff_plus = D_j_plus_half / (h_j_next * (h_j_prev + h_j_next)/2)
        coeff_minus = D_j_minus_half / (h_j_prev * (h_j_prev + h_j_next)/2)
        
        main_diag[j] = coeff_plus + coeff_minus + alpha_n[n]**2 * D_values[j]
        upper_diag[j] = -coeff_plus
        lower_diag[j-1] = -coeff_minus

    h_0 = z[1] - z[0]
    main_diag[0] = D_values[0]/h_0 + kappa * h_0/2
    upper_diag[0] = -D_values[0]/h_0

    main_diag[-1] = 1.0
    upper_diag[-1] = 0.0
    lower_diag[-1] = 0.0

    main_diag += 1e-10 * D0  # Estabilización aumentada

    A = diags(
        diagonals=[main_diag, upper_diag, lower_diag],
        offsets=[0, 1, -1],
        shape=(N_z, N_z),
        format='csr'
    )
    
    eigenvalues, eigenvectors = eigsh(
        A,
        k=m_max,
        which='SA',
        tol=1e-6,
        maxiter=200000,
        ncv=150
    )
    
    eigenvalues_dict[n] = np.clip(eigenvalues, 1e-14, None)  # Forzar positividad
    eigenvectors_dict[n] = eigenvectors

# =============================================
# 5. Cálculo de Coeficientes A_nm Corregido
# =============================================
def compute_A_nm(n):
    A_nm_list = []
    Z_norms = np.zeros(m_max)
    
    for m in range(m_max):
        integrand = eigenvectors_dict[n][:, m]**2 / D_values
        Z_norms[m] = simpson(integrand, z)
    
    Z_m_z0 = CubicSpline(z, eigenvectors_dict[n])(z0)
    J0_rho0 = jv(0, alpha_n[n] * rho0)
    
    for m in range(m_max):
        A_nm_val = (J0_rho0 * Z_m_z0[m] / 
                   (2 * np.pi * rho0 * norm_J0_sq[n] * Z_norms[m]))
        A_nm_list.append(A_nm_val)
    
    return A_nm_list

A_nm = Parallel(n_jobs=-1)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

# =============================================
# 6. Cálculo de G(t) Corregido
# =============================================
def compute_G(t, N_quad=15):
    rho_quad, w_rho = np.polynomial.legendre.leggauss(N_quad)
    rho_quad = 0.5 * R * (rho_quad + 1)
    w_rho *= 0.5 * R
    
    z_quad, w_z = np.polynomial.legendre.leggauss(N_quad)
    z_quad = z.min() + (z.max() - z.min())*0.5*(z_quad + 1)
    w_z *= 0.5*(z.max() - z.min())
    
    # Precalcular todas las interpolaciones (n,m)
    Z_interps = [
        [CubicSpline(z, eigenvectors_dict[n][:, m]) 
        for m in range(m_max)
    ] for n in range(n_modes_radial)
    ]
    
    def integrand(i, j):
        rho_i, z_i = rho_quad[i], z_quad[j]
        f_ij = (3*z_i**2 - rho_i**2)/(rho_i**2 + z_i**2)**2.5
        sum_term = 0.0
        
        for n in range(n_modes_radial):
            J0 = jv(0, alpha_n[n] * rho_i)
            for m in range(m_max):
                Z_m = Z_interps[n][m](z_i)
                decay = np.exp(-(alpha_n[n]**2 * D0 + eigenvalues_dict[n][m]) * t)
                
                if decay > 1e-12:
                    sum_term += A_nm[n][m] * J0 * Z_m * decay
        
        return w_rho[i] * w_z[j] * f_ij * sum_term
    
    results = Parallel(n_jobs=-1)(delayed(integrand)(i,j) 
                for i in range(N_quad) for j in range(N_quad))
    
    G = (5/(16*np.pi)) * np.sum(results)
    return max(G, 1e-20)

# =============================================
# 7. Ejecución Principal y Gráficas
# =============================================
if __name__ == "__main__":
    plt.figure(figsize=(8,5))
    plt.plot(z*1e9, D_values/D0, 'b-', lw=1.5)
    plt.xlabel('z (nm)')
    plt.ylabel('D(z)/D₀')
    plt.title('Perfil de Difusión')
    plt.grid(ls='--', alpha=0.7)
    plt.savefig('Dz_profile_v2.pdf', bbox_inches='tight')
    plt.close()
    
    plt.figure(figsize=(8,5))
    for n in range(3):
        evals = eigenvalues_dict[n][:20]
        plt.semilogy(evals, 'o-', label=f'n={n}')
    plt.xlabel('Modo Axial')
    plt.ylabel('Autovalor (s⁻¹)')
    plt.title('Espectro de Autovalores')
    plt.legend()
    plt.grid(ls='--', alpha=0.7)
    plt.savefig('eigenvalues_v2.pdf', bbox_inches='tight')
    plt.close()
    
    t_values = np.logspace(-10, -0.01, 100)
    G_values = Parallel(n_jobs=-1)(delayed(compute_G)(t) for t in t_values)
    
    np.savetxt('G_vs_t_v2.dat', np.c_[t_values, G_values], 
               header='Tiempo(s) G(t)', fmt='%.3e')
    
    plt.figure(figsize=(8,5))
    plt.loglog(t_values, G_values, 'ko-', label='Simulación')
    plt.plot(t_values, 1e-12*t_values**-1.5, 'r--', label='t⁻¹·⁵')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('G(t)')
    plt.title('Función de Correlación')
    plt.legend()
    plt.grid(ls='--', alpha=0.7)
    plt.savefig('Gt_result_v2.pdf', bbox_inches='tight')
    plt.close()

In [2]:
evals

array([1.00000000e-14, 1.00000000e-14, 1.00000000e-14, 1.00000000e-14,
       1.00000000e-14, 1.00000000e-14, 1.00000000e-14, 1.00000000e-14,
       1.00000000e-14, 1.00000000e-14, 1.00000000e-14, 1.00000000e-14,
       1.00000000e-14, 1.21896719e+11, 3.34757853e+11, 4.17079949e+11,
       6.15749584e+11, 6.71252330e+11, 9.47626274e+11, 1.13149258e+12])